# SAKSHI — head-detector fine-tune (RF-DETR Nano on SCUT-HEAD)

Runs **in Google Colab on a T4 GPU only** (project rule: no training on the local machine).
Outputs, all written to `/content/out/` and copied to `MyDrive/sakshi_models/`:

| File | What |
|---|---|
| `rfdetr_nano_head.onnx` | Fine-tuned head detector (1 class: head) for seated rooms |
| `rfdetr_nano_coco.onnx` | Pretrained COCO detector, used for `person` on door cameras |
| `head_metrics.json` | AP on SCUT-HEAD test splits, head-count MAE, chosen score threshold |

Licences: RF-DETR Nano code + weights Apache-2.0. SCUT-HEAD has no licence file → research use only;
the trained weights are used for the prototype and that limitation is recorded in `docs/DATASETS.md`.


In [ ]:
!nvidia-smi -L
!pip install -q "rfdetr[train,onnx]==1.11.2" gdown==5.2.0


## 1. Download SCUT-HEAD (Part A: classroom CCTV, Part B: web images)


In [ ]:
import os, glob, subprocess
os.makedirs('/content/scut', exist_ok=True)
%cd /content/scut
# File IDs from https://github.com/HCIILAB/SCUT-HEAD-Dataset-Release (README download links)
for name, fid in [('PartA', '1yaOF9os5wPVNNG4GVzNyULLVe74vdrBE'), ('PartB', '1LZ_KlTPStDEcqycfqUkDkqQ-aNMMC3cl')]:
    if not glob.glob(f'*{name}*'):
        subprocess.run(['gdown', '--fuzzy', f'https://drive.google.com/open?id={fid}', '-O', f'{name}.archive'], check=False)
!ls -la /content/scut


In [ ]:
# Extract whatever archive format the release uses (zip / rar / tar).
!pip install -q patool==4.0.1 && apt-get -qq install -y unrar >/dev/null
import patoolib, os
for f in os.listdir('.'):
    if f.endswith('.archive'):
        patoolib.extract_archive(f, outdir=f.replace('.archive',''), verbosity=-1)
!find /content/scut -maxdepth 3 -type d | head -30


## 2. Pascal VOC → COCO in RF-DETR's folder layout (`train/ valid/ test_A/ test_B/`)


In [ ]:
import xml.etree.ElementTree as ET, json, shutil, random
from pathlib import Path

def voc_root(part):
    # Locate the folder containing JPEGImages/ Annotations/ ImageSets/
    for p in Path(f'/content/scut/{part}').rglob('JPEGImages'):
        return p.parent
    raise FileNotFoundError(part)

def ids(root, split):
    f = root / 'ImageSets' / 'Main' / f'{split}.txt'
    return [l.strip() for l in f.read_text().split() if l.strip()]

def to_coco(items, out_dir):
    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    images, anns, aid = [], [], 1
    for iid, (root, stem) in enumerate(items, 1):
        x = ET.parse(root / 'Annotations' / f'{stem}.xml').getroot()
        w, h = int(x.find('size/width').text), int(x.find('size/height').text)
        fname = f'{root.parent.name}_{stem}.jpg'
        shutil.copy(root / 'JPEGImages' / f'{stem}.jpg', out / fname)
        images.append({'id': iid, 'file_name': fname, 'width': w, 'height': h})
        for o in x.findall('object'):
            b = o.find('bndbox')
            x1, y1, x2, y2 = (float(b.find(k).text) for k in ('xmin', 'ymin', 'xmax', 'ymax'))
            if x2 <= x1 or y2 <= y1: continue
            anns.append({'id': aid, 'image_id': iid, 'category_id': 1, 'bbox': [x1, y1, x2 - x1, y2 - y1],
                         'area': (x2 - x1) * (y2 - y1), 'iscrowd': 0}); aid += 1
    json.dump({'images': images, 'annotations': anns, 'categories': [{'id': 1, 'name': 'head', 'supercategory': 'person'}]},
              open(out / '_annotations.coco.json', 'w'))
    return len(images), len(anns)

A, B = voc_root('PartA'), voc_root('PartB')
trainval = [(A, s) for s in ids(A, 'trainval')] + [(B, s) for s in ids(B, 'trainval')]
random.Random(0).shuffle(trainval)
n_val = len(trainval) // 10
D = Path('/content/heads')
print('train', to_coco(trainval[n_val:], D / 'train'))
print('valid', to_coco(trainval[:n_val], D / 'valid'))
print('test_A (classroom CCTV)', to_coco([(A, s) for s in ids(A, 'test')], '/content/heads_testA/test'))
print('test_B (web)', to_coco([(B, s) for s in ids(B, 'test')], '/content/heads_testB/test'))
# RF-DETR's evaluate() wants a test/ split inside dataset_dir: use Part A test (the classroom domain).
shutil.copytree('/content/heads_testA/test', D / 'test', dirs_exist_ok=True)


## 3. Fine-tune RF-DETR Nano (resolution 512: small back-row heads, still CPU-friendly)


In [ ]:
from rfdetr import RFDETRNano
EPOCHS = 30
model = RFDETRNano()
model.train(dataset_dir='/content/heads', epochs=EPOCHS, batch_size='auto', lr=1e-4,
            resolution=512, output_dir='/content/train_heads', checkpoint_interval=5,
            notes={'project': 'SAKSHI SIH26245', 'data': 'SCUT-HEAD A+B trainval'})


## 4. Evaluate: AP on test splits + head-count MAE, pick the score threshold on *valid* (not test)


In [ ]:
import numpy as np, json, glob
from PIL import Image
best = sorted(glob.glob('/content/train_heads/checkpoint_best*.pth'))[0]
print('best checkpoint:', best)
head = RFDETRNano(pretrain_weights=best, resolution=512)

def counts(split_dir, thr_list):
    ann = json.load(open(f'{split_dir}/_annotations.coco.json'))
    gt = {i['id']: 0 for i in ann['images']}
    for a in ann['annotations']: gt[a['image_id']] += 1
    pred = {t: [] for t in thr_list}; truth = []
    for im in ann['images']:
        det = head.predict(Image.open(f"{split_dir}/{im['file_name']}").convert('RGB'), threshold=min(thr_list))
        for t in thr_list: pred[t].append(int((det.confidence >= t).sum()))
        truth.append(gt[im['id']])
    return np.array(truth), {t: np.array(v) for t, v in pred.items()}

thrs = [round(x, 2) for x in np.arange(0.2, 0.75, 0.05)]
gt_v, pv = counts('/content/heads/valid', thrs)
mae_v = {t: float(np.abs(pv[t] - gt_v).mean()) for t in thrs}
thr = min(mae_v, key=mae_v.get)
res = {'chosen_threshold': thr, 'valid_mae_by_threshold': mae_v, 'resolution': 512, 'epochs': EPOCHS}
for name, d in [('test_A_classroom', '/content/heads_testA/test'), ('test_B_web', '/content/heads_testB/test')]:
    gt, p = counts(d, [thr])
    res[name] = {'images': len(gt), 'mean_heads': float(gt.mean()), 'count_mae': float(np.abs(p[thr] - gt).mean()),
                 'count_mae_pct': float((np.abs(p[thr] - gt) / np.maximum(gt, 1)).mean() * 100)}
try:
    res['test_A_coco_metrics'] = {k: float(v) for k, v in head.evaluate(dataset_dir='/content/heads', split='test').items()}
except Exception as e:
    res['test_A_coco_metrics'] = f'evaluate() failed: {e!r}'
res['gpu'] = !nvidia-smi --query-gpu=name --format=csv,noheader
os.makedirs('/content/out', exist_ok=True)
json.dump(res, open('/content/out/head_metrics.json', 'w'), indent=2, default=str)
print(json.dumps(res, indent=2, default=str))


## 5. Export both ONNX models (fp32, static batch 1) and save to Drive


In [ ]:
import shutil
p = head.export(output_dir='/content/export_head', format='onnx', fp16=False)
shutil.copy(p, '/content/out/rfdetr_nano_head.onnx')
coco = RFDETRNano()  # pretrained COCO weights (Apache-2.0)
p = coco.export(output_dir='/content/export_coco', format='onnx', fp16=False)
shutil.copy(p, '/content/out/rfdetr_nano_coco.onnx')
!cd /content/out && sha256sum * | tee SHA256SUMS && ls -la


In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/sakshi_models && cp /content/out/* /content/drive/MyDrive/sakshi_models/
!cd /content && zip -qr sakshi_models.zip out
files.download('/content/sakshi_models.zip')
